# 09 - Persiapan Dataset Training IndoBERTweet (Binary & 3-Class)

Tahap 3.1.4 lanjutan pada proposal TA. Menggabungkan hasil labeling 2 annotator (tahap 08) +
resolusi disagreement, normalisasi teks sesuai konvensi IndoBERTweet, lalu split train/val/test +
5-fold CV. Disiapkan untuk DUA skema sekaligus (binary & 3-class) dari baris komentar yang SAMA,
supaya hasil kedua skema bisa dibandingkan adil.

**Alur (1 cell per tahap):**
1. `align_labelers()` -- sejajarkan `labels_shafwa.csv` & `labels_khalila.csv` baris-per-baris.
2. `merge_disagreements()` -- baris yang 2 annotator SETUJU langsung pakai label itu
   (`label_source='agree'`); yang tidak setuju diisi dari `disagreements_label_final.csv` hasil
   resolusi manual tahap 08 (`label_source='resolved'`).
3. `normalize_text()` -- konvensi IndoBERTweet: `@mention`->`@USER`, URL->`HTTPURL`, emoji
   didemojize jadi teks lalu dikembalikan ke kalimat.
4. `split_and_prepare()` -- StratifiedKFold 7-bagian (test~14%/val~14%/train~72%) + assignment
   fold 5-CV, stratified berdasarkan `label_final`. Untuk tiap skema: hitung `class_weight`,
   simpan `dataset_final.csv` lengkap + train/val/test siap tokenisasi + `config.json`
   (label2id, class_weights, MAX_LEN hasil cek distribusi panjang token P95).

**Konsolidasi dari:** `02_IndoBert/03_finetunning_model/01_persiapan_data.ipynb`

**Catatan eksekusi:** ringan (pandas + sklearn, tanpa scraping/LLM) kecuali unduh tokenizer
HuggingFace (sekali, kecil) untuk hitung MAX_LEN. Skip otomatis kalau output kedua skema sudah ada.

In [ ]:
import json
import re
from pathlib import Path

import emoji
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# -------------------------------------------------------------------------
# KONFIGURASI
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
LABELING_DIR = FINAL_OUTPUT_ROOT / "08_labeling_sentimen"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "09_persiapan_dataset_training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_SHAFWA = LABELING_DIR / "labels_shafwa.csv"
LABELS_KHALILA = LABELING_DIR / "labels_khalila.csv"
DISAGREEMENTS_FINAL = LABELING_DIR / "disagreements_label_final.csv"

SPLIT_BY = "comment"  # 'comment' | 'influencer' (satu influencer tidak terpecah antar split)
SEED = 42
N_FOLDS = 5
N_SPLIT_PARTS = 7  # 7 bagian -> test ~14%, val ~14%, train ~72%
USE_S1_PREPROCESS = True
MODEL_NAME = "indolem/indobertweet-base-uncased"

SCHEME_DEFS = {
    "binary": ["non-negatif", "negatif"],
    "3class": ["negatif", "netral", "positif"],
}

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
USER_RE = re.compile(r"@\w+")
print("Config siap.")

## STEP 1 - Sejajarkan 2 annotator baris-per-baris

In [ ]:
def align_labelers() -> pd.DataFrame:
    s = pd.read_csv(LABELS_SHAFWA)
    k = pd.read_csv(LABELS_KHALILA)
    if len(s) != len(k):
        raise ValueError(f"Jumlah baris beda: {len(s)} vs {len(k)}")

    same_row = ((s["video_id"].astype(str).values == k["video_id"].astype(str).values)
                & (s["comment_text_raw"].fillna("").values == k["comment_text_raw"].fillna("").values))
    if not same_row.all():
        raise ValueError(f"{(~same_row).sum()} baris tidak sejajar antara dua file labeler")

    df = s[["username", "video_id", "comment_text_raw", "comment_text_clean"]].copy()
    df["label_shafwa"] = s["label_manual"].str.lower().str.strip()
    df["label_khalila"] = k["label_manual"].str.lower().str.strip()
    print(f"[Align] {len(df)} baris tersejajar dari 2 annotator")
    return df


df = align_labelers()

## STEP 2 - Gabung agree + resolved disagreement

In [ ]:
def merge_disagreements(df: pd.DataFrame) -> pd.DataFrame:
    if not DISAGREEMENTS_FINAL.exists():
        raise FileNotFoundError(
            f"{DISAGREEMENTS_FINAL} tidak ada. Resolusi manual disagreement (tahap 08) belum selesai."
        )

    dis = pd.read_csv(DISAGREEMENTS_FINAL)
    dis["label_shafwa"] = dis["label_shafwa"].str.lower().str.strip()
    dis["label_khalila"] = dis["label_khalila"].str.lower().str.strip()
    dis["label_final"] = dis["label_final"].str.lower().str.strip()

    key = ["comment_text_clean", "label_shafwa", "label_khalila"]
    conflict = dis.groupby(key)["label_final"].nunique()
    if not (conflict == 1).all():
        raise ValueError("Ada komentar yang sama tapi label_final berbeda di disagreements_label_final.csv")
    dis_map = dis.drop_duplicates(key)[key + ["label_final"]]

    df = df.merge(dis_map, on=key, how="left", validate="many_to_one")

    is_agree = df["label_shafwa"] == df["label_khalila"]
    df.loc[is_agree, "label_final"] = df.loc[is_agree, "label_shafwa"]
    df["label_source"] = np.where(is_agree, "agree", "resolved")

    missing = df["label_final"].isna().sum()
    if missing > 0:
        raise ValueError(f"{missing} komentar disagreement belum punya label_final")

    ok = (df["label_final"].eq(df["label_shafwa"]) | df["label_final"].eq(df["label_khalila"])
          | df["label_final"].eq("netral"))
    if not ok.all():
        raise ValueError("Ada label_final yang melanggar aturan (bukan salah satu label annotator / netral)")

    print(df["label_source"].value_counts().to_string())
    return df


df = merge_disagreements(df)

## STEP 3 - Normalisasi teks (konvensi IndoBERTweet)

In [ ]:
def _normalize_tweet(text) -> str:
    text = "" if pd.isna(text) else str(text)
    text = URL_RE.sub("HTTPURL", text)
    text = USER_RE.sub("@USER", text)
    text = emoji.demojize(text, language="id")
    text = re.sub(r":([^:\s]+):", r" \1 ", text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    source_text = "comment_text_raw"
    base = df[source_text].where(
        df[source_text].notna() & df[source_text].astype(str).str.strip().ne(""),
        df["comment_text_clean"],
    )
    if USE_S1_PREPROCESS:
        df["text_model"] = base.map(_normalize_tweet)
    else:
        df["text_model"] = df["comment_text_clean"].fillna("").astype(str).str.strip().str.lower()

    empty = df["text_model"].str.len() == 0
    print(f"[Normalize] Teks kosong setelah normalisasi: {empty.sum()} (dibuang)")
    return df[~empty].reset_index(drop=True)


df = normalize_text(df)

## STEP 4 - Split train/val/test + fold, simpan per skema

In [ ]:
def _assign_parts(data: pd.DataFrame, n_parts: int, seed: int, strat_col: str) -> np.ndarray:
    if SPLIT_BY == "influencer":
        cv = StratifiedGroupKFold(n_splits=n_parts, shuffle=True, random_state=seed)
        it = cv.split(data, data[strat_col], groups=data["username"])
    else:
        cv = StratifiedKFold(n_splits=n_parts, shuffle=True, random_state=seed)
        it = cv.split(data, data[strat_col])
    part = np.full(len(data), -1)
    for i, (_, te) in enumerate(it):
        part[te] = i
    return part


def split_and_prepare(df: pd.DataFrame) -> None:
    parts = _assign_parts(df, N_SPLIT_PARTS, SEED, "label_final")
    df["split"] = np.select([parts == 0, parts == 1], ["test", "val"], default="train")
    df["fold"] = _assign_parts(df, N_FOLDS, SEED, "label_final")

    n_neg_test = ((df["split"] == "test") & (df["label_final"] == "negatif")).sum()
    n_neg_all = (df["label_final"] == "negatif").sum()
    print(f"[Split] Negatif total: {n_neg_all} | di test: {n_neg_test}")
    if n_neg_test < 30:
        print("[Split] PERHATIAN: komentar negatif di test < 30, F1-negatif tidak stabil. "
              "Pertimbangkan cross-validation (kolom fold) dan/atau tambah data negatif (tahap 10).")

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    lens = df["text_model"].map(lambda t: len(tok.encode(t, add_special_tokens=True)))
    p95 = int(np.ceil(np.percentile(lens, 95)))
    max_len = int(min(128, int(np.ceil(p95 / 8) * 8)))
    print(f"[Split] Panjang token P95={p95} -> MAX_LEN={max_len} "
          f"(terpotong: {(lens > max_len).mean() * 100:.1f}%)")

    cols_full = ["username", "video_id", "comment_text_raw", "comment_text_clean", "text_model",
                 "label_shafwa", "label_khalila", "label_final", "label_source",
                 "label_name", "label", "split", "fold"]

    for scheme_name, label_names in SCHEME_DEFS.items():
        scheme_dir = OUTPUT_DIR / scheme_name
        if (scheme_dir / "train.csv").exists():
            print(f"[{scheme_name}] Sudah ada -> {scheme_dir} (skip)")
            continue

        d = df.copy()
        label2id = {n: i for i, n in enumerate(label_names)}
        id2label = {i: n for n, i in label2id.items()}
        d["label_name"] = (np.where(d["label_final"] == "negatif", "negatif", "non-negatif")
                            if scheme_name == "binary" else d["label_final"])
        d["label"] = d["label_name"].map(label2id)

        scheme_dir.mkdir(parents=True, exist_ok=True)
        d[cols_full].to_csv(scheme_dir / "dataset_final.csv", index=False)
        for part in ["train", "val", "test"]:
            d.loc[d["split"] == part, ["text_model", "label", "label_name", "username", "video_id"]] \
                .to_csv(scheme_dir / f"{part}.csv", index=False)

        train = d[d["split"] == "train"]
        classes = np.array(sorted(d["label"].unique()))
        w = compute_class_weight("balanced", classes=classes, y=train["label"])
        class_weights = {int(c): float(x) for c, x in zip(classes, w)}

        dist = d["label_name"].value_counts().reindex(label_names)
        print(f"=== {scheme_name} ===")
        print(pd.DataFrame({"jumlah": dist, "persen": (dist / len(d) * 100).round(1)}))
        print("class_weight:", {id2label[c]: round(x, 3) for c, x in class_weights.items()})

        config = {
            "model_name": MODEL_NAME, "label_scheme": scheme_name, "split_by": SPLIT_BY,
            "seed": SEED, "use_s1_preprocess": USE_S1_PREPROCESS, "label2id": label2id,
            "id2label": {str(v): k_ for k_, v in label2id.items()},
            "class_weights": class_weights, "max_len": max_len, "n_total": int(len(d)),
        }
        (scheme_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"[{scheme_name}] Tersimpan -> {scheme_dir}")


if (OUTPUT_DIR / "binary" / "train.csv").exists() and (OUTPUT_DIR / "3class" / "train.csv").exists():
    print(f"[09] Sudah ada -> {OUTPUT_DIR} (skip)")
else:
    split_and_prepare(df)